---

title: Slotted Union Finds 
---

Blabber

The slotted egraph is a data structure to deal with named variables in terms, which show up all over the place.

There are a number of different extra features in the slotted union find that we've been discovering generalize in a number of ways.







# Sets of Terms with Named Variables

The slotted egraph and the slotted union find are designed to be talking more or less about the specific model of sets of named terms.

An eclass in an egraph can be thought of as either
1. a set of terms like `{3, 2+1, 1+2, 1+1+1}`
2. Some value in an underlying domain `3`

Both are useful perspectives but it is really important to be aware of which one you are using.

We have an interpretation of terms into `Int` that we can use to connect the two pictures

```
[[3]] = 3
[[?a + ?b]] = [[?a]] + [[?b]]
```

In the slotted egraph, there is first class support for named variables with renaming.

Picture 1 thinks about terms with names in them `{var(x) + 2, var(x) + 1 + 1, 2 + var(x), ...}`. NOTE! This is not the _same_ set as `{var(y) + 2, var(y) + 1 + 1, 2 + var(y), ...}`, but they are related by renaming. Variables kind of matter, but also kind of don't in a subtle way. If you build a data structure that understands renaming natively, these two things can share a lot of memory storage and you can have certain renaming related inferences happen automatically, akin to how rebuilding automatically propagates congruence closure.

Each eid is intended to mean some set of terms. This may not be a finite set of terms in an egraph. Loops in the egraph can represent infinite sets of terms in a finite data like `{0, 0*1, 0*0*1, 0*0*0*1,...}`

In the the slotted egraph, each eid represents a set of terms with names. A renamed eid represents applying a renaming to every term in the set.

```
m * var(x) = m[x]
m * f(a,b,...) = f(m*a, m*b, ...)
m * {t0,t1,t2, ...} = {m*t0, m*t1, m*t2, ...}
```
So for example
```
[x -> y, ...] * {var(x) + 2, var(x) + 1 + 1, 2 + var(x), ...} = {var(y) + 2, var(y) + 1 + 1, 2 + var(y), ...}
```

The equation of `var(x) * 0 = 0` results in an eid that represents `{var(x) * 0, 0}`. If two equivalence classes have a common term in them, they should be merged. 

If we renamed `[x -> y, ...] * {var(x) * 0, 0}` we get the set `{var(y) * 0, 0}`. This is a different set. BUT, these sets share the term `0`. Hence they should be merged into `{var(y) * 0, var(x) * 0, 0}`. We can continue to play this game for every variable. We end up with the infinite set `{var(y) * 0, var(x) * 0, var(z) * 0, ..., 0}` a set that contains a term for every possible variable. 
A union-like over approximation free variable analysis on this set would return ALL variables.

However, what this is demonstrating from a different perspective, is that the variable in some sense does not matter. It is _redundant_. If looking up the variable in the environment returns any random junk, that gets cleared by multiplying by zero and the result is just zero. Typically, languages error out when you ask for an variable that is not bound in the environment, but that is not the only possible semantics, and not the one in mind here.
Under this semantics, the under approximation / intersection free variable analysis is the more relevant one. This is the information associated with the slotted egraph's "public slots". The public slots are the slots that are actually relevant to the result of the expression, not the slots that appear syntactically in the expression. There is no information flow from the slots outside of public slots to the result. 


A similar thing happens when the set of terms possesses a self symmettry under renaming. 
Let's say we start with `f(var(x), var(y)) = var(x) + var(y)` This results in an eid that means the set of named term
`{f(var(x), var(y)), var(x) + var(y)}`
. If we then assert `var(x) + var(y) = var(y) + var(x)` we get the set `{f(var(x), var(y)). var(x) + var(y), var(y) + var(x)}`. In an ordinary semantic for an egraph, we'd be done at this point. But if we apply a renaming `[x -> y, y -> x]` to this set, we get the new distinct set `{f(var(y), var(x)). var(x) + var(y), var(y) + var(x)}` These two sets share common terms and there fore must be merged resulting in `{f(var(x), var(y)), f(var(y), var(x)), var(x) + var(y), var(y) + var(x)}`. Now the partition of terms is invariant under renamings.

Let's suppose we were actually using python sets to store terms. It would be a memory saving to not store all the symmettry related versions of the terms in that final set. We could instead store the reduced/factored form
`{f(var(x), var(y)),var(y) + var(x)}` alongside a renaming symmetry group `{[x -> x, y -> y], [x -> y, y -> x]}`, which one can use computational group theory methods compress and manipulate even further. Symettry groups grow fast (`n!`) but can be encoded in polynomial space.


```
a ~ b
-------
m*a ~ m*b



m*a ~ m*b   m bijective
-----------------------
a ~ b


a ~ b
-------------   offset union find
n + a ~ n + b

a ~ b 
-------- this is not generally true. This is the condition what it means to be "good" metaf
metaf(a) ~ metaf(b)


x - y = 0
----------
x - x = 0

```



`(Name -> Real) -> Real`

```
[[3]] env = 3
[[?a + ?b]] env = ([[?a]] env) + ([[?b]] env)
[[var(i)]] env = env i 
```

`(Name -> Option Real) -> Real`


`[[m * e]] env = [[e]] (env . m)` NO 

m :: Name -> Option Name 

`(m * [[e]]) env = [[e]] (env . m)` defines `*`

{var(x) + 3, var(x) + 2}

`[[m]]
{x -> y, y -> x}
[[m]] = fun name => if name in m then m[x] else name`



[[Gamma |- t]] 



In [ ]:
type Name = str
type Env = dict[Name, object]
type Value = Callable[[Env], object]
def interp(t : Term) -> Value:
    match t:
        case Var(name):
            return lambda env: env[name]
        case Lit(value):
            return lambda env: value
        case App("+", args):
            return lambda env: sum(interp(arg)(env) for arg in args)

def rename_term(r : Renaming, t : Term) -> Term:
    ...

def rename_env(r : Renaming, env : Env) -> Env:
    ...

def rename_value(r : Renaming, v : Value) -> Value:
    return lambda env: v(rename_env(r.inv(), env))

# interp(rename_term(t)) == rename_value(interp(t))

# t1 ~ t2  ==> interp(t1) == interp(t2)
# m*t1 ~ m*t2 ==> interp(m*t1) == interp(m*t2) ====> rename_value(m, interp(t1)) == rename_value(m, interp(t2))


# Basic Union Find

In [ ]:
from dataclasses import dataclass, field

@dataclass
class UF():
    parents : list[int] = field(default_factory=list)
    def makeset(self):
        a = len(self.parents)
        self.parents.append(a)
        return a
    def find(self, a):
        while self.parents[a] != a:
            a = self.find(self.parents[a])
        return a
    def union(self, a, b):
        a,b = self.find(a), self.find(b)
        if a != b:
            self.parents[a] = b

# Bijective Renamings are a Group

Bijective renamings are basically permutations of the names. Permutations are a group. There is a known variant of union finds that carry group annotations between different ids.

A nice example of this is an integer offset union find that can express relationships like `3 + x = 4 + y`

NOTE! Even though `x` and `y` are related, they actually quite explicitly ARE NOT EQUAL. They are rather "equal up to" an offset or can be said to be in the same "G-class" in the sense there is a known group element G (offset +1) that relates them.

This point gets more confusing when people start talking about alpha equivalence. Structurally similar terms with different names are NOT enerally equal to each other. But they are related. Once one closes the terms, the semantic meaning of different namings do tend to become equal, but even then, differently named terms are NOT equal in general. They are related (by renaming).





In [ ]:
class OffSetUF:
    ...

In [ ]:
type Slot = int

@dataclass(frozen=True)
class Renaming:
    map: frozenset[tuple[Slot, Slot]]

    def __repr__(self):
        return "[" + ", ".join(f"{a} -> {b}" for (a, b) in self.map) + "]"

    @classmethod
    def of_list(cls, lst: list[tuple[Slot, Slot]]):
        return cls(frozenset(lst))

    @classmethod
    def identity(cls, slots: list[Slot]):
        return cls.of_list([(s, s) for s in slots])

    def rev(self):
        return Renaming.of_list([(b, a) for (a, b) in self.map])

    def keys(self):
        return [a for (a, b) in self.map]

    def values(self):
        return [b for (a, b) in self.map]

    def get(self, key: Slot):
        for a, b in self.map:
            if a == key:
                return b
        return key

    def __getitem__(self, key: Slot):
        for a, b in self.map:
            if a == key:
                return b
        raise KeyError(key)

    def compose(self, q: "Renaming") -> "Renaming":
        """
        self : X -> Y
        q : Y -> Z
        self @ q : X -> Z
        """

        return Renaming.of_list([(a, q[b]) for (a, b) in self])

    def compose_partial(self, q):
        return Renaming.of_list([(a, q[b]) for (a, b) in self if b in q.keys()])

    def __matmul__(self, q):
        # self is renaming X -> Y
        # q is renaming Y -> Z
        # returns renaming X -> Z
        return self.compose(q)

    def __mul__(self, eid: Union[int, "RenamedId"]) -> "RenamedId":
        # renaming * eid
        # takes eid : Y
        # self : Y -> Z
        # returns Z
        if isinstance(eid, int):
            return RenamedId(eid, self)
        elif isinstance(eid, RenamedId):
            # eid.id : X
            # eid.renaming : X -> Y
            # eid : Y
            # self : Y -> Z
            return RenamedId(eid.id, eid.renaming.compose(self))
        else:
            raise TypeError(eid)

    def __iter__(self):
        return iter(self.map)



@dataclass(frozen=True)
class RenamedId:
    id: int
    renaming: Renaming

    def __repr__(self):
        return f"{self.renaming} * id{self.id}"

    def slots(self) -> set[Slot]:
        return set(self.renaming.values())


We mentioned earlier that we need to store symmettries of eids. We can do this in the most brute force way of just storing all elements of the symmettry subgroup.



In [ ]:
@dataclass
class Group:
    perms: set[Renaming]

    def __init__(self, elems: set[Slot]):
        identity = Renaming.of_list(list(zip(elems, elems)))
        self.perms = {identity}

    def contains(self, p: Perm):
        return p in self.perms

    def add(self, p: Perm):
        self.perms.add(p)
        self.complete()

    def complete(self):
        while True:
            cnt = len(self.perms)
            newperms = []
            for p in self.perms:
                newperms.append(p.rev())
            for p1 in self.perms:
                for p2 in self.perms:
                    newperms.append(p1.compose(p2))
            self.perms.update(newperms)
            if cnt == len(self.perms):
                break

    def orbit(self, slot: Slot) -> set[Slot]:
        return {p.get(slot) for p in self.perms}

    def __len__(self):
        return len(self.perms)


# Slotted Union Find

The slotted union find follows the rough lines of a group union find + extra goodies for symmettry and redundancy propagation.

NOTE: The group elements in the _symmettries_ are describing an equality. The group elements in the parents table are describing an actual change of names that changes the meaning of the eid.

Something very useful to get the various points where an inverse of the renaming is necessary is to sort of "type check " everything. Every `RenamedId` has certain public_slots associated with it. Renamings have to have these slots as their domain to make any sense of applying a renaming to a RenamedId. In addition, a union between two renamedIds that have differing slots is ok, it results in redundancy propagation.


In [ ]:
@dataclass
class SlottedUF:
    uf: list[RenamedId] = field(
        default_factory=list
    )  # Id -> RenamedId. Really wanted RenamedId -> RenamedId conceptually. But doing it this way is deduplicating in exactly the way we want to be deduplicating.
    public_slots: dict[int, set[Slot]] = field(default_factory=dict)
    symmetries: dict[int, Group] = field(default_factory=dict)


    def makeset(self, arity: int) -> RenamedId:
        # Make a renamed id with identity trasnformation
        slots = list(range(arity))
        n = len(self.uf)
        eid = RenamedId(n, Renaming.of_list([(a, a) for a in slots]))
        self.uf.append(eid)
        self.public_slots[n] = set(slots)
        self.symmetries[n] = Group(set(slots))
        return eid

    def makeset_slots(self, slots: list[Slot]) -> RenamedId:
        eid = self.makeset(len(slots))
        return Renaming.of_list(zip(eid.slots(), slots)) * eid

    def find(self, ma: RenamedId) -> RenamedId:
        # U[m*a] = m*U[id*a] = m*(m'*b)
        # find[m*a] = m*find[id*a] = m*uf[a] = m*(m'*b) = (m o m')*b
        assert isinstance(ma, RenamedId)
        rename = ma.renaming
        a = ma.id  # This is kind of a canonization step. Turning a renamed thing into a "canonical" named version of it
        while True:
            mb = self.uf[a]
            rename = mb.renaming @ rename
            if mb.id == a:
                return RenamedId(id=a, renaming=rename)
            a = mb.id

    def shrink_slots(self, a: RenamedId, remaining_slots: set[Slot]):
        pslots = self.public_slots[a.id]
        assert pslots >= remaining_slots
        if pslots == remaining_slots:
            return  # nothing to do
        """
        We need to lose all the the slots related to the losing slots by symmetry
        """
        G = self.symmetries[a.id]
        losing_slots = {slot for s in pslots - remaining_slots for slot in G.orbit(s)}
        remaining_slots = remaining_slots - losing_slots
        b = self.makeset(len(remaining_slots))

        # a.id : X
        # a.m : X -> Y
        # a : Y
        # remaining_slots : Y
        # pslots : Y
        # b : Z
        # need renaming : Z -> X
        r = Renaming.of_list(list(zip(b.renaming.values(), remaining_slots)))  # Z -> Y
        renaming = r @ a.renaming.rev()  # Z -> X
        self.uf[a.id] = RenamedId(renaming=renaming, id=b.id)
        for perm in self.symmetries[a.id].perms:
            self.symmetries[b.id].add(
                renaming.compose_partial(perm.compose_partial(renaming.rev()))
            )

    def union(self, a: RenamedId, b: RenamedId) -> bool:
        # union(self, m1*a1, m2*a2)
        # union(self, U[m1*a1], U[m2*a2])
        # union(self, m3 * a3, m4 * a4)
        # union(self, id * a3, (m3^-1 o m4) * a4)
        # ufunion(self, a3, (m3^-1 o m4) * a4)
        #
        while True:
            a, b = self.find(a), self.find(b)
            aslots = set(a.renaming.values())
            bslots = set(b.renaming.values())
            if aslots != bslots:
                self.shrink_slots(a, aslots & bslots)
                self.shrink_slots(b, aslots & bslots)
                # redundant slots
            else:
                break
            #    a.rev()[]
        a, b = self.find(a), self.find(b)
        if a.id != b.id:
            # TODO: merge symmetries
            m = b.renaming.compose(a.renaming.rev())
            # a : Z
            # a.id : X
            # b.id : Y
            # a.renaming : X -> Z
            # b.renaming : Y -> Z
            # m = b.renaming @ a.renaming.rev : Y -> X
            # b : Z
            # perm : X -> X
            # m @ perm @ m.rev : Y -> Y
            for perm in self.symmetries[a.id].perms:
                self.symmetries[b.id].add(m @ perm @ m.rev())
            self.uf[a.id] = RenamedId(id=b.id, renaming=m)
            return True
        else:
            self.symmetries[a.id].add(a.renaming.rev() @ b.renaming)
            return False

    def is_eq(self, a: RenamedId, b: RenamedId) -> bool:
        a = self.find(a)
        b = self.find(b)
        if set(a.renaming.values()) != set(b.renaming.values()):
            return False
        # but actually symmetries
        # a.id : X
        # a.renaming : X -> Y
        # b.id : X
        # b.renaming : X -> Y
        # a.renaming @ b.renaming.rev() : X -> X
        return (
            a.id == b.id
            and a.renaming @ b.renaming.rev() in self.symmetries[a.id].perms
        )

# Offset and Factor Union Find

In [ ]:
from dataclasses import dataclass, field
from fractions import Fraction

type Id = tuple[Fraction,int]
@dataclass
class MulUF:
    parents: list[Id] = field(default_factory=list)
    def makeset(self) -> Id:
        id = (Fraction(1), len(self.parents))
        self.parents.append(id)
        return id
    def find(self, x: Id) -> Id:
        m,xid = x
        while True:
            m1, yid = self.parents[xid]
            if xid == yid:
                assert m1 == 1
                return (m,xid)
            else:
                m *= m1
                xid = yid
    def union(self, x : Id, y : Id) -> Id | None:
        mx, xid = self.find(x)
        my, yid = self.find(y)
        if xid == yid:
            if mx == my:
                return None
        if mx == 0 and my == 0:
        


# Multiplication By Zero as A Union Find Annotation


# Bits and Bobbles

---
title: Slotted Union Finds
---

Co-written with Rudi Schneider

E-graphs are data structure for storing many equations and present a solution to the phase ordering problem in compilers. Slotted e-graphs are a proposal for compacting alpha equivalent terms in an e-graph. Many modelling use cases want summation $\Sigma$ or lambda terms and alpha equivalence is part of the story there.

I (Philip) have struggled to understand what is going on in slotted. I've sort of felt they don't fit into my way of thinking about things.

A general idea pump in e-graphs is

- hashcons + unionfind = egraph
- egraph - hashcons = union find
- egraph - union find = hash cons

It can be simpler to consider removing pieces from an egraph and look at what kind of hash cons or union find you get. Likewise exotic hash conses or union find lead to exotic e-graphs.

This blog post is about the slotted union find. In turn, we agree that it is kind of a nice abstraction for construction slotted e-graphs

# Basic Union Find
A union find is a forest where the pointers point up to their parents. This makes it easy to merge trees without eagerly updating all the children.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class UF():
    parents : list[int] = field(default_factory=list)
    def makeset(self):
        a = len(self.parents)
        self.parents.append(a)
        return a
    def find(self, a):
        while self.parents[a] != a:
            a = self.find(self.parents[a])
        return a
    def union(self, a, b):
        a,b = self.find(a), self.find(b)
        if a != b:
            self.parents[a] = b

    

# Group Union Finds

It is possible to add annotations to union finds. This gives them a bit more expressive power. They can be quite cheap sometimes. This may be useful in some applications.

Annotations could appear
- at roots
- at ids
- on edges

A noted form of this is the ability to add group elements to edges. The multiplication of the group is used to accumulate the annotation as you traverse up the tree in `find`. The identity is useful to give a null annotation. Inverse is useful in `union` when you need to find a way to isolate a annotated identifier to a bare identifier `ma * a = mb * b ===> a = ma^-1 * mb * b`  


Here is probably the simplest version of this, using the integers `Z` considered as an offset group


In [ ]:
type Offset = int
type Id = int
type OId = tuple[Offset, Id]
class OffsetUF():
    parents : list[OId] = field(default_factory=list)
    def makeset(self) -> OId:
        a = len(self.parents)
        oid = (0,a)
        self.parents.append(oid)
        return oid
    def find(self, oid: OId) -> OId:
        (off, a) = oid
        while True:
            (off1, b) = self.parents[a]
            if b == a:
                assert off1 == 0
                return (off, a)
            off += off1
            a = b 
    def union(self, a: OId, b: OId):
        (offa, ida) = a
        (offb, idb) = b
        ida, idb = self.find((offa, ida))[1], self.find((offb, idb))[1]
        if ida != idb:
            self.parents[ida] = (offa - offb, idb)
        elif ida == idb:
            if offa != offb:
                raise Exception("offset mismatch")



# Permutation Groups

In an e-graph, one of the interpretations is that eclasses represent a set of equivalent terms `{f(bar,biz), f(biz,baz)}. In the slotted e-graph, these terms contain special names that are intended to be alpha invariant  `{f(a, b), g(a,a,b)}`. This set is _related_ but _not equal_ to another set `{f(a1, b1), g(a1,a1,b1)}` by a renaming process of some kind. This is akin to how if we know that two things are offset related `x = y + 42` this does not mean they are necessarily _equal_. We want to factor out this renaming process so that we can compactly represent the many renaming related but nevertheless distinct sets of terms.

Bijective renamings can be thought of as permutations. Permutations form a group.



In [ ]:
class Perm():
    

# Symmetries



# Redundancies

# Bits and Bobbles

My blog posts
Rudi's
Slotted Paper and talks
github repo

group union find paper
ed kmett
groupoid blog post


If a thinning union find is baking in a (witness carrying) notion of sublist into union find annotations, then a slotted union find is baking in key renaming and sub dictionaries into union find annotations. I like this story because it is completely divorced from contexts or variables or names. Totally mundane. Yes, you can use lists or dictionaries to talk about contexts / environments.
So maybe the right way to do inequalities is to have witness certificates to the inequality?
